# A/B-тест: оформление заказа в интернет-магазине

**Учебный кейс на полностью синтетических данных.** 30 000 участников, вариант A (control) / B (treatment). Все цифры можно воспроизвести скриптами репозитория.

**Задача:** понять, изменило ли упрощение checkout долю пользователей, которые оформили заказ. Вторично смотрим чистую выручку на пользователя и возвраты.

## Шаг 1. Загружаем данные
Каждая строка = один рандомизированный пользователь, а не заказ. Поэтому **конверсия = число покупателей / число всех назначенных пользователей**.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from scipy.stats import chisquare
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analysis import validate, compute

df = pd.read_csv(ROOT / "data" / "checkout_experiment_synthetic.csv")
print("Rows, columns:", df.shape)
print(df.head(5).to_string(index=False))

Rows, columns: (30000, 8)
user_id assigned_at variant  device traffic_source  converted  order_value_rub  refunded
u000001  2026-09-03 control  mobile        organic          0                0         0
u000002  2026-09-13 control  mobile           paid          1             1690         0
u000003  2026-09-19 control desktop           paid          0                0         0
u000004  2026-09-12 control  mobile        organic          0                0         0
u000005  2026-09-02 control  mobile           paid          0                0         0


## Шаг 2. Проверяем корректность событий
- уникальность `user_id`;
- только варианты A/B;
- нельзя получить возврат, если не было покупки;
- нет положительной суммы заказа у непокупателя.

`validate` остановит выполнение, если эти инварианты нарушены.

In [2]:
validate(df)
print("Unique users:", df.user_id.nunique())
print("Missing values by column:")
print(df.isna().sum().to_string())

Unique users: 30000
Missing values by column:
user_id            0
assigned_at        0
variant            0
device             0
traffic_source     0
converted          0
order_value_rub    0
refunded           0


## Шаг 3. Проверяем группы (SRM)
Гипотеза по назначению: **50% control / 50% treatment**. Для сильного дисбаланса используем chi-square; тревога SRM, если p < 0.001. SRM проверяется отдельно от значимости продуктового эффекта.

In [3]:
assigned = df.variant.value_counts().reindex(["control", "treatment"])
srm = chisquare(assigned.to_numpy(), f_exp=[len(df) / 2] * 2)
print("Users per arm:", assigned.to_dict())
print("SRM p-value:", float(srm.pvalue))

Users per arm: {'control': 14975, 'treatment': 15025}
SRM p-value: 0.7728299926844475


## Шаг 4. Метрика: конверсия
Гипотезы: **H0: CR_B = CR_A**. **H1: CR_B ≠ CR_A**. Заранее выбираем двусторонний тест и уровень значимости 5%.

In [4]:
summary = df.groupby("variant").agg(
    users=("user_id", "count"),
    purchases=("converted", "sum"),
    gross_revenue_rub=("order_value_rub", "sum"),
    refunds=("refunded", "sum"),
)
summary["conversion"] = summary["purchases"] / summary["users"]
print(summary.to_string())
cr_a = summary.loc["control", "conversion"]
cr_b = summary.loc["treatment", "conversion"]
print(f"Effect: {(cr_b-cr_a)*100:+.2f} percentage points, relative {(cr_b/cr_a-1)*100:+.2f}%")

           users  purchases  gross_revenue_rub  refunds  conversion
variant                                                            
control    14975       1602            4217009       81    0.106978
treatment  15025       1849            4865306       76    0.123062
Effect: +1.61 percentage points, relative +15.03%


## Шаг 5. Статистическая проверка
Сравниваем **две независимые доли** z-тестом. Для размера эффекта используем 95% ДИ Newcombe. Если p < 0.05, отклоняем H0, но это не равно бизнес-выгоде.

In [5]:
x_b, x_a = summary.loc["treatment", "purchases"], summary.loc["control", "purchases"]
n_b, n_a = summary.loc["treatment", "users"], summary.loc["control", "users"]
z, p = proportions_ztest([x_b, x_a], [n_b, n_a], alternative="two-sided")
ci = confint_proportions_2indep(x_b, n_b, x_a, n_a, compare="diff", method="newcomb")
print(f"z={z:.3f}, p={p:.6f}")
print(f"95% CI for B-A: [{ci[0]*100:.2f}, {ci[1]*100:.2f}] pp")

z=4.365, p=0.000013
95% CI for B-A: [0.89, 2.33] pp


## Шаг 6. Экономика и защитные метрики
Вычитаем **полностью** возвращённые покупки из выручки (упрощение данных). Чистая выручка **на всех назначенных пользователей**, а не только покупателей. Доля возвратов условна по покупателям и не доказывает отсутствие риска.

In [6]:
df["net_revenue_rub"] = df.order_value_rub * (1 - df.refunded)
economics = df.groupby("variant").agg(
    users=("user_id", "count"),
    net_revenue=("net_revenue_rub", "sum"),
    refunds=("refunded", "sum"),
    buyers=("converted", "sum"),
)
economics["net_revenue_per_user"] = economics.net_revenue / economics.users
economics["refund_rate_given_purchase"] = economics.refunds / economics.buyers
print(economics.to_string())

           users  net_revenue  refunds  buyers  net_revenue_per_user  refund_rate_given_purchase
variant                                                                                         
control    14975      4002963       81    1602            267.309716                    0.050562
treatment  15025      4666241       76    1849            310.565125                    0.041103


## Шаг 7. Вывод и ограничения
Скрипт `src/analysis.py` сохраняет два графика, JSON и подробный `reports/findings.md`. Полная версия содержит также бутстрэп интервал для выручки и валидацию SQL-агрегатов.

**Бизнес-интерпретация:** есть ли убедительный эффект конверсии; какой размер эффекта; что неизвестно о марже, возвратах и стоимости внедрения. Данные учебные — нельзя переносить эффект на реальных покупателей.

In [7]:
result = compute(pd.read_csv(ROOT / "data" / "checkout_experiment_synthetic.csv"))
print("Decision for simulated data:", result["recommendation"])
print("Refund rate A/B:",
      f"{result['refund_rate_given_purchase_control']:.2%}",
      f"{result['refund_rate_given_purchase_treatment']:.2%}")
print("See reports/findings.md for the automatically generated report and charts.")

Decision for simulated data: Consider a staged rollout with refund and revenue monitoring
Refund rate A/B: 5.06% 4.11%
See reports/findings.md for the automatically generated report and charts.
